# 📈 ML Forecasting, Experiment Tracking og MLflow

Denne notebook-en demonstrerer hvordan du kjører **ML-baserte tidsserieprognoser** for KOSTRA-kontoer og sporer eksperimenter, parametere og prediksjonsavvik ved hjelp av **MLflow** i VS Code.

### Hvorfor MLflow?
* **Eksperimentstyring:** Sammenligne lineær run-rate, Random Forest, XGBoost og Prophet.
* **Model Registry & Drift Monitoring:** Overvåke når regnskapstallene avviker fra ML-prognosene (*Concept Drift*), noe som automatisk utløser et **Early Warning**-signal.

In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import os

# Sett lokal MLflow tracking URI (SQLite database)
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('KOSTRA_Avvik_Forecasting')
print('✅ MLflow lokal tracking initialisert')

## 1. Trening og Logging av ML-Prognosemodell

In [ ]:
# Syntetisk/Eksempel på treningssett for KOSTRA Vikarutgifter
X_train = np.array([[1], [2], [3], [4], [5], [6], [7], [8]]) # Måneder 1-8
y_train = np.array([45, 50, 52, 60, 85, 90, 95, 120])       # Regnskap kNOK

n_estimators = 50
max_depth = 3

with mlflow.start_run(run_name='RandomForest_Vikar_Prognose'):
    # Logg hyperparametere
    mlflow.log_param('n_estimators', n_estimators)
    mlflow.log_param('max_depth', max_depth)
    mlflow.log_param('target_account', '102013_Vikarer')
    
    # Tren modell
    model = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)
    
    # Evaluering
    predictions = model.predict(X_train)
    rmse = np.sqrt(mean_squared_error(y_train, predictions))
    mae = mean_absolute_error(y_train, predictions)
    
    # Logg evalueringskriterier
    mlflow.log_metric('rmse', rmse)
    mlflow.log_metric('mae', mae)
    
    # Logg modellen
    mlflow.sklearn.log_model(model, 'model')
    
    print(f'✅ MLflow Run fullført | RMSE: {rmse:.2f} | MAE: {mae:.2f}')

## 2. Drift Monitoring & Early Warning Trigger

Når ny månedlig regnskapsdata kommer inn, sammenligner vi ML-modellens prognose med de faktiske tallene for å avdekke **Data Drift**.

In [ ]:
# Ny måned (Måned 9 - September)
faktisk_september = 155.0 # kNOK
ml_prediksjon_september = model.predict([[9]])[0]

avvik_pct = ((faktisk_september - ml_prediksjon_september) / ml_prediksjon_september) * 100

print(f'🤖 ML Prediksjon September: {ml_prediksjon_september:.1f} kNOK')
print(f'📊 Faktisk Regnskap September: {faktisk_september:.1f} kNOK')
print(f'⚠️ Avvik i forhold til ML-modell: {avvik_pct:.1f}%')

if avvik_pct > 20.0:
    print('🔴 EARLY WARNING: Regnskapet avviker mer enn 20% fra ML-prognosen! Modell må om-trenes eller tiltak iverksettes.')
else:
    print('🟢 MODEL IN BOUNDS: Avviket er innenfor akseptabel toleranse.')